# 0. **Import Libraries**

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
!python -m pip install scikit-learn

In [4]:
!python -m pip install --upgrade pip

In [5]:
!python -m pip install skrebate

In [1]:
import pandas as pd
import numpy as np
import json

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from skrebate import ReliefF

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier            # Leaf-wise gradient boosting, best for high-dim tabular
from sklearn.linear_model import LogisticRegression  # Linear baseline

from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import os
import joblib

import warnings
warnings.filterwarnings('ignore')  # suppress LightGBM & sklearn verbose warnings

# 1. **Data Collection & EDA**
---
## [APS](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks)
- Positive class --> Failure from APS
- Negative class --> Other failure (not APS)
---

### Columns name is **Anonymized**

In [2]:
# Load data
train_df = pd.read_csv(r'C:\College\DS\Project\Dataset\aps_failure_training_set.csv', skiprows=20, na_values='na')
test_df = pd.read_csv(r'C:\College\DS\Project\Dataset\aps_failure_test_set.csv', skiprows=20, na_values='na')

## **EDA**

In [4]:
train_df.head()

,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


In [5]:
test_df.head()

,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,60,0.0,20.0,12.0,0.0,0.0,0.0,0.0,0.0,...,1098.0,138.0,412.0,654.0,78.0,88.0,0.0,0.0,0.0,0.0
1,neg,82,0.0,68.0,40.0,0.0,0.0,0.0,0.0,0.0,...,1068.0,276.0,1620.0,116.0,86.0,462.0,0.0,0.0,0.0,0.0
2,neg,66002,2.0,212.0,112.0,0.0,0.0,0.0,0.0,0.0,...,495076.0,380368.0,440134.0,269556.0,1315022.0,153680.0,516.0,0.0,0.0,0.0
3,neg,59816,NaN,1010.0,936.0,0.0,0.0,0.0,0.0,0.0,...,540820.0,243270.0,483302.0,485332.0,431376.0,210074.0,281662.0,3232.0,0.0,0.0
4,neg,1814,NaN,156.0,140.0,0.0,0.0,0.0,0.0,0.0,...,7646.0,4144.0,18466.0,49782.0,3176.0,482.0,76.0,0.0,0.0,0.0


In [6]:
print(f'Train data shape: {train_df.shape}')
print(f'Test data shape: {test_df.shape}')

Train data shape: (60000, 171)
Test data shape: (16000, 171)


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 171 entries, class to eg_000
dtypes: float64(169), int64(1), object(1)
memory usage: 78.3+ MB


In [8]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Columns: 171 entries, class to eg_000
dtypes: float64(169), int64(1), object(1)
memory usage: 20.9+ MB


In [9]:
print(f'Duplicate rows in train data: {train_df.duplicated().sum()}')
print(f'Duplicate rows in test data: {test_df.duplicated().sum()}')

Duplicate rows in train data: 0
Duplicate rows in test data: 0


In [10]:
print('Missing values in train data:')
train_df.isnull().sum().sort_values(ascending= False)

Missing values in train data:


br_000    49264
bq_000    48722
bp_000    47740
bo_000    46333
ab_000    46329
          ...  
cj_000      338
ci_000      338
bt_000      167
aa_000        0
class         0
Length: 171, dtype: int64

In [11]:
print('\nMissing values in test data:')
test_df.isnull().sum().sort_values(ascending= False)


Missing values in test data:


br_000    13129
bq_000    12981
bp_000    12721
bo_000    12376
ab_000    12363
          ...  
cj_000       86
ci_000       86
bt_000       28
aa_000        0
class         0
Length: 171, dtype: int64

In [12]:
# NaN values percentile in training data
train_df['class'].value_counts(normalize= True) * 100

class
neg    98.333333
pos     1.666667
Name: proportion, dtype: float64

In [13]:
# # NaN values percentile in test data
test_df['class'].value_counts(normalize= True) * 100

class
neg    97.65625
pos     2.34375
Name: proportion, dtype: float64

# 2. Cleaning Functions

In [3]:
def clean_data(df):
    df = df.copy()
    # df['class'] = df['class'].map({'pos': 1, 'neg': 0})
    return df


def drop_bad_columns(df, threshold=0.6):
    missing_percent = df.isnull().mean()
    cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
    df = df.drop(columns=cols_to_drop)
    return df, cols_to_drop


def split_data(df):
    X = df.drop(columns=['class'])
    y = df['class']
    return X, y

# 3. Feature Engineering

In [4]:
def remove_corr_keep_best(df, y, threshold=0.95):
    y_numeric = (y == 'pos').astype(int)

    corr_matrix = df.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    target_corr = df.corrwith(y_numeric).abs()

    to_drop = set()

    for col in upper.columns:
        if col in to_drop:  # Skip already-dropped columns
            continue

        high_corr = upper[col][upper[col] > threshold].index

        for other_col in high_corr:
            if other_col in to_drop:
                continue

            # compare importance using correlation with target
            if target_corr[col] < target_corr[other_col]:
                to_drop.add(col)
            else:
                to_drop.add(other_col)

    df_reduced = df.drop(columns=list(to_drop))

    print(f"  Removed {len(to_drop)} correlated features | Remaining: {df_reduced.shape[1]}")

    return df_reduced, list(to_drop)

In [5]:
# def feature_engineering(X):
#     X = X.copy()

#     X['row_mean'] = X.mean(axis=1)
#     X['row_std']  = X.std(axis=1)
#     X['row_min']  = X.min(axis=1)
#     X['row_max']  = X.max(axis=1)
#     X['row_range']  = X['row_max'] - X['row_min']
#     X['row_median'] = X.median(axis=1)
#     X['row_skew']   = X.skew(axis=1)

#     return X

In [6]:
def combined_feature_selection(
    X, y,
    n_features=50,
    w_mi=0.4,
    w_rf=0.4,
    w_relief=0.2,
    random_state=42
):
    feature_names = X.columns

    # Encode y temporarily to numeric — required by ReliefF and safer for MI
    y_encoded = (y == 'pos').astype(int).values

    # ── 1. Mutual Information ─────────────────────────
    mi_scores = mutual_info_classif(X, y_encoded, random_state=random_state)
    mi_scores = pd.Series(mi_scores, index=feature_names)

    # ── 2. Random Forest Importance ──────────────────
    rf = RandomForestClassifier(
        random_state=random_state,
        class_weight='balanced',
        n_jobs=-1
    )
    rf.fit(X, y_encoded)
    rf_scores = pd.Series(rf.feature_importances_, index=feature_names)

    # ── 3. ReliefF ───────────────────────────────────
    sample_idx = np.random.RandomState(random_state).choice(
        len(X), size=min(5000, len(X)), replace=False
    )
    relief = ReliefF(n_neighbors=50)  # Reduced from 100 to 50 to save RAM
    relief.fit(X.values[sample_idx], y_encoded[sample_idx])
    relief_scores = pd.Series(relief.feature_importances_, index=feature_names)

    # ── 4. Normalize scores ──────────────────────────
    def normalize(series):
        # Separate scaler per method to avoid state leakage
        arr = MinMaxScaler().fit_transform(series.values.reshape(-1, 1)).flatten()
        return pd.Series(arr, index=feature_names)

    mi_norm     = normalize(mi_scores)
    rf_norm     = normalize(rf_scores)
    relief_norm = normalize(relief_scores)

    # ── 5. Combine scores ────────────────────────────
    combined_score = (
        w_mi     * mi_norm +
        w_rf     * rf_norm +
        w_relief * relief_norm
    )

    # ── 6. Select top features ───────────────────────
    n_features  = min(n_features, len(feature_names))
    top_features = combined_score.nlargest(n_features).index.tolist()

    print(f"  Selected {len(top_features)} features from {len(feature_names)}")

    return {
        "selected_features": top_features,
        "combined_score"   : combined_score.sort_values(ascending=False),
        "mi"               : mi_norm,
        "rf"               : rf_norm,
        "relief"           : relief_norm
    }

# 4. Preprocessing Pipeline (TRAIN)

In [7]:
def preprocess_train(df):
    # ── Clean ──────────────────────────────────────────────
    df = clean_data(df)

    # ── Drop high-missing columns ──────────────────────────
    df, dropped_cols = drop_bad_columns(df)

    # ── Split X / y ────────────────────────────────────────
    X, y = split_data(df)
    feature_names = list(X.columns)

    # ── Imputation ─────────────────────────────────────────
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    X = pd.DataFrame(X, columns=feature_names)  # restore names

    # ── Variance selection ─────────────────────────────────
    var_selector = VarianceThreshold(0.01)
    X = var_selector.fit_transform(X)
    feature_names = [feature_names[i]
                     for i, s in enumerate(var_selector.get_support()) if s]
    X = pd.DataFrame(X, columns=feature_names)

    # ── Scaling ────────────────────────────────────────────
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = pd.DataFrame(X, columns=feature_names)  # restore names

    # ── Remove Correlated Features ─────────────────────────
    X, dropped_corr = remove_corr_keep_best(X, y, threshold=0.95)
    feature_names = list(X.columns)

    # ── Feature Selection (MI + RF + ReliefF) ──────────────
    result = combined_feature_selection(X, y, n_features=50)
    selected_features = result["selected_features"]
    X = X[selected_features]

    return {
        "X"               : X,
        "y"               : y,
        "imputer"         : imputer,
        "scaler"          : scaler,
        "dropped_cols"    : dropped_cols,
        "var_selector"    : var_selector,
        "selected_features": selected_features,
        "dropped_corr"    : dropped_corr
    }

# 5. Preprocessing (TEST)

In [8]:
def preprocess_test(df, train_out):
    imputer         = train_out["imputer"]
    scaler          = train_out["scaler"]
    dropped_cols    = train_out["dropped_cols"]
    var_selector    = train_out["var_selector"]
    selected_features = train_out["selected_features"]
    dropped_corr    = train_out["dropped_corr"]

    df = clean_data(df)
    df = df.drop(columns=dropped_cols, errors='ignore')

    X, y = split_data(df)
    feature_names = list(X.columns)

    # ── Imputation ─────────────────────────────────────────
    X = imputer.transform(X)
    X = pd.DataFrame(X, columns=feature_names)

    # ── Variance selection ─────────────────────────────────
    X = var_selector.transform(X)
    feature_names = [feature_names[i]
                     for i, s in enumerate(var_selector.get_support()) if s]
    X = pd.DataFrame(X, columns=feature_names)

    # ── Scaling ────────────────────────────────────────────
    X = scaler.transform(X)
    X = pd.DataFrame(X, columns=feature_names)

    # ── Drop correlated features ───────────────────────────
    X = X.drop(columns=dropped_corr, errors='ignore')

    # ── Select final features ──────────────────────────────
    X = X.reindex(columns=selected_features, fill_value=0)

    return X, y

# 6. Training + Evaluation

In [9]:
def train_and_evaluate(X_train, y_train, X_test, y_test):

    # ── SMOTE ──────────────────────────────────────────────
    # Encode y to numeric temporarily — required by LightGBM & safer for SMOTE
    y_train_enc = (y_train == 'pos').astype(int)
    y_test_enc  = (y_test  == 'pos').astype(int)

    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train_enc)

    # ── Models ─────────────────────────────────────────────
    models = {
        'Random Forest': RandomForestClassifier(
            random_state=42,
            # Removed class_weight — SMOTE already balanced the classes
        ),
        'LightGBM': LGBMClassifier(
            random_state=42,
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=63,
            verbose=-1,
        ),
        'Logistic Regression': LogisticRegression(
            random_state=42,
            max_iter=1000,
            solver='lbfgs'
        )
    }

    # ── Summary table to compare all models at the end ────────────────
    summary = []

    for model_name, model in models.items():
        print(f'\n' + '='*50)
        print(f'  Model: {model_name}')
        print('='*50)

        # ── Fit ───────────────────────────────────────────
        model.fit(X_train_res, y_train_res)

        # ── Save trained model ───────────────────────────
        safe_name = model_name.lower().replace(" ", "_")
        joblib.dump(model, f"models/{safe_name}.pkl")
        
        # ── Prediction ────────────────────────────────────
        y_pred = model.predict(X_test)

        # ── Evaluation ────────────────────────────────────
        acc = accuracy_score(y_test_enc, y_pred)
        rec = recall_score(y_test_enc, y_pred)
        f1  = f1_score(y_test_enc, y_pred)
        training_metrics = {
            "accuracy": acc,
            "recall": rec,
            "f1": f1,
            "cm": confusion_matrix(y_test_enc, y_pred).tolist()  # Convert to list for JSON serialization
        }
        with open(f"models/{safe_name}_metrics.json", "w") as f:
            json.dump(training_metrics, f, indent=4)

        print(f'Accuracy : {acc:.4f}')
        print(f'Recall   : {rec:.4f}')
        print(f'F1 Score : {f1:.4f}')
        print('\nConfusion Matrix:')
        print(confusion_matrix(y_test_enc, y_pred))
        print('\nClassification Report:')
        print(classification_report(y_test_enc, y_pred))

        # Collect for summary
        summary.append({
            'Model'   : model_name,
            'Accuracy': round(acc, 4),
            'Recall'  : round(rec, 4),
            'F1'      : round(f1,  4)
        })

    # Print comparison table at the end
    print('\n' + '='*50)
    print('  SUMMARY')
    print('='*50)
    print(pd.DataFrame(summary).to_string(index=False))

# 7. RUN EVERYTHING

In [10]:
# ==========================================
# RUN EVERYTHING
# ==========================================

# ── Step 1: Preprocess Train ───────────────
print("⏳ Preprocessing training data...")
train_out = preprocess_train(train_df)
X_train   = train_out["X"]
y_train   = train_out["y"]
print("✅ Train preprocessing done!")

# ── Step 2: Preprocess Test ────────────────
print("\n⏳ Preprocessing test data...")
# [EDITED] pass train_out dict directly — contains all fitted artifacts
X_test, y_test = preprocess_test(test_df, train_out)
print("✅ Test preprocessing done!")

# ── Step 3: Sanity check ───────────────────
print(f"\n📊 Train shape    : {X_train.shape}")
print(f"📊 Test shape     : {X_test.shape}")
print(f"📊 Target values  : {y_train.unique()}")   # should show ['neg' 'pos']
# [ADDED] class distribution check — important for imbalanced dataset
print(f"📊 Class dist (train): {y_train.value_counts().to_dict()}")
print(f"📊 Class dist (test) : {y_test.value_counts().to_dict()}")

# ── Step 4: Train & Evaluate all models ────
print("\n🚀 Starting model training & evaluation...")
train_and_evaluate(X_train, y_train, X_test, y_test)

⏳ Preprocessing training data...
  Removed 21 correlated features | Remaining: 139
  Selected 50 features from 139
✅ Train preprocessing done!

⏳ Preprocessing test data...
✅ Test preprocessing done!

📊 Train shape    : (60000, 50)
📊 Test shape     : (16000, 50)
📊 Target values  : ['neg' 'pos']
📊 Class dist (train): {'neg': 59000, 'pos': 1000}
📊 Class dist (test) : {'neg': 15625, 'pos': 375}

🚀 Starting model training & evaluation...

  Model: Random Forest
Accuracy : 0.9896
Recall   : 0.8187
F1 Score : 0.7862

Confusion Matrix:
[[15526    99]
 [   68   307]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     15625
           1       0.76      0.82      0.79       375

    accuracy                           0.99     16000
   macro avg       0.88      0.91      0.89     16000
weighted avg       0.99      0.99      0.99     16000


  Model: LightGBM
Accuracy : 0.9915
Recall   : 0.8533
F1 Score : 0.8247

Confusion

In [11]:
os.makedirs("models", exist_ok=True)

artifacts = artifacts = {
    "imputer": train_out["imputer"],
    "scaler": train_out["scaler"],
    "dropped_cols": train_out["dropped_cols"],
    "var_selector": train_out["var_selector"],
    "selected_features": train_out["selected_features"],
    "dropped_corr": train_out["dropped_corr"],
}

joblib.dump(artifacts, "models/preprocess_artifacts.pkl")

['models/preprocess_artifacts.pkl']